# 04 – CẢI TIẾN TRÍCH XUẤT THÔNG TIN HÓA ĐƠN SROIE

Notebook này là sản phẩm chính của phần 04. Toàn bộ luật Baseline cần dùng và luật cải tiến được trình bày trực tiếp trong Notebook; không cần tạo/import `improved.py`. Notebook chỉ sử dụng dữ liệu Train và Validation, không nạp Test và không chỉnh sửa Ground Truth.

**Phương pháp được chọn:** cải tiến cách trích xuất `company` bằng cách xếp hạng các dòng đầu hóa đơn; `date`, `address`, `total` giữ nguyên luật Baseline vì các thử nghiệm thay thế chưa cho thấy lợi ích trên Train.

## 1. Cấu trúc thư mục và đầu vào

Đặt Notebook trong thư mục gốc dự án. Notebook cần ba file dữ liệu sau (đường dẫn có thể sửa ở cell kế tiếp):

- `outputs/ocr/train_validation_ocr.csv`
- `outputs/ground_truth/train_ground_truth.csv`
- `outputs/ground_truth/validation_ground_truth.csv`

Các hàm Baseline và Improved được viết ngay trong Notebook, vì vậy không cần thêm file `improved.py` hoặc import module cải tiến riêng.

In [1]:
from pathlib import Path
import re, unicodedata
from datetime import datetime
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
# Nếu Notebook được mở từ thư mục con, thử tìm thư mục dự án ở cấp cha.
if not (PROJECT_DIR / "outputs").exists():
    candidates = [PROJECT_DIR.parent, PROJECT_DIR / "project"]
    for candidate in candidates:
        if (candidate / "outputs").exists():
            PROJECT_DIR = candidate
            break

OCR_PATH = PROJECT_DIR / "outputs/ocr/train_validation_ocr.csv"
GT_TRAIN = PROJECT_DIR / "outputs/ground_truth/train_ground_truth.csv"
GT_VAL = PROJECT_DIR / "outputs/ground_truth/validation_ground_truth.csv"
OUT_DIR = PROJECT_DIR / "outputs/improved"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for path in [OCR_PATH, GT_TRAIN, GT_VAL]:
    assert path.exists(), f"Không tìm thấy file đầu vào: {path}. Hãy kiểm tra lại đường dẫn."

FIELDS = ["company", "date", "address", "total"]

# ===== BASELINE: giữ nguyên logic từ src/baseline.py của nhóm =====
MONTHS = r"(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*"
DATE_PATTERNS = [
    r"\b\d{1,2}\s*[/\-.]\s*\d{1,2}\s*[/\-.]\s*\d{2,4}\b",
    r"\b\d{4}\s*[/\-.]\s*\d{1,2}\s*[/\-.]\s*\d{1,2}\b",
    r"\b\d{1,2}[\s\-]*" + MONTHS + r"[\s\-,]*\d{2,4}\b",
]
DATE_RE = re.compile("|".join(f"(?:{pattern})" for pattern in DATE_PATTERNS), re.IGNORECASE)
AMOUNT_RE = re.compile(r"\d[\d,]*[.,]\s?\d{2}\b")
TOTAL_KEYS = ["GRAND TOTAL", "TOTAL ROUNDED", "ROUNDED TOTAL", "NET TOTAL", "TOTAL AMOUNT", "AMOUNT DUE", "AMOUNT PAYABLE", "TOTAL"]
TOTAL_EXCLUDE = re.compile(r"SUB|EXCL|QTY|QUANTITY|SAVING|DISCOUNT|POINT|TOTAL\s*GST|TAX", re.IGNORECASE)
ADDR_KW = re.compile(r"\b(JALAN|JLN|TAMAN|TMN|LOT|NO|BANDAR|KAWASAN|SELANGOR|JOHOR|KUALA LUMPUR|KL|PETALING|SHAH ALAM|KLANG|CHERAS|PAHANG|PENANG|WILAYAH)\b", re.IGNORECASE)
POSTCODE = re.compile(r"\b\d{5}\b")
ADDR_STOP = re.compile(r"\b(TEL|FAX|GST|INVOICE|RECEIPT|DATE|CASH|BILL|TAX|DOC|CASHIER)\b", re.IGNORECASE)

def _get_lines(ocr_text):
    if not isinstance(ocr_text, str):
        return []
    return [line.strip() for line in ocr_text.split("\n") if line.strip()]

def _find_company(lines):
    for i, line in enumerate(lines[:5]):
        if sum(char.isalpha() for char in line) >= 3:
            return line, i
    return None, -1

def _find_date(lines):
    for line in lines:
        if "DATE" in line.upper():
            match = DATE_RE.search(line)
            if match:
                return match.group(0).strip()
    match = DATE_RE.search("\n".join(lines))
    return match.group(0).strip() if match else None

def _find_total(lines):
    for key in TOTAL_KEYS:
        for i, line in enumerate(lines):
            upper = line.upper()
            if key in upper and not TOTAL_EXCLUDE.search(upper):
                amounts = AMOUNT_RE.findall(line)
                if not amounts and i + 1 < len(lines):
                    amounts = AMOUNT_RE.findall(lines[i + 1])
                if amounts:
                    return amounts[-1].replace(" ", "")
    return None

def _find_address(lines, company_idx):
    if company_idx < 0:
        return None
    parts = []
    for line in lines[company_idx + 1:company_idx + 9]:
        if ADDR_STOP.search(line):
            break
        if ADDR_KW.search(line) or POSTCODE.search(line):
            parts.append(line)
    return " ".join(parts) if parts else None

def extract_baseline(ocr_text):
    lines = _get_lines(ocr_text)
    company, company_idx = _find_company(lines)
    return {
        "company": company,
        "date": _find_date(lines),
        "address": _find_address(lines, company_idx),
        "total": _find_total(lines),
    }

print("Thư mục dự án:", PROJECT_DIR)
print("OCR:", OCR_PATH)
print("Ground Truth Train:", GT_TRAIN)
print("Ground Truth Validation:", GT_VAL)

Thư mục dự án: c:\Users\Admin\HOA_DON_AI
OCR: c:\Users\Admin\HOA_DON_AI\outputs\ocr\train_validation_ocr.csv
Ground Truth Train: c:\Users\Admin\HOA_DON_AI\outputs\ground_truth\train_ground_truth.csv
Ground Truth Validation: c:\Users\Admin\HOA_DON_AI\outputs\ground_truth\validation_ground_truth.csv


## 2. Kiểm tra OCR và Ground Truth

Chỉ nạp hai split Train/Validation. Kiểm tra key trùng, split ngoài phạm vi và số lượng mẫu trước khi chạy.

In [2]:
ocr = pd.read_csv(OCR_PATH, dtype={"key": str}).fillna({"ocr_text": ""})
gt_train = pd.read_csv(GT_TRAIN, dtype=str).fillna("")
gt_val = pd.read_csv(GT_VAL, dtype=str).fillna("")
gt = pd.concat([gt_train, gt_val], ignore_index=True)
assert ocr.key.is_unique and gt.key.is_unique, "Phát hiện key trùng"
assert set(ocr.split.unique()) <= {"train", "validation"}, "OCR có split ngoài Train/Validation"
assert len(ocr[ocr.split=="train"]) == len(gt_train)
assert len(ocr[ocr.split=="validation"]) == len(gt_val)
assert set(ocr.key) == set(gt.key), "Key OCR và Ground Truth không khớp"
df = ocr.merge(gt[["key"]+FIELDS], on="key", how="inner", validate="one_to_one")
print("OCR:", ocr.shape, "| GT:", gt.shape)
print(ocr.split.value_counts())
print("OCR status:", ocr.status.value_counts(dropna=False).to_dict())
print("GT rỗng theo trường:", {f:int(df[f].fillna("").str.strip().eq("").sum()) for f in FIELDS})

OCR: (626, 4) | GT: (626, 5)
split
train         501
validation    125
Name: count, dtype: int64
OCR status: {'success': 626}
GT rỗng theo trường: {'company': 0, 'date': 0, 'address': 1, 'total': 1}


## 3. Chuẩn hóa để đánh giá

Các hàm dưới đây chỉ phục vụ tính Exact Match; giá trị OCR dự đoán thô và Ground Truth gốc vẫn được lưu riêng trong CSV.

In [3]:
DATE_FORMATS = ["%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y", "%d/%m/%y", "%d-%m-%y", "%d.%m.%y", "%Y-%m-%d", "%Y/%m/%d", "%d %b %Y", "%d%b%Y", "%d-%b-%Y", "%d %b %y", "%d %B %Y"]
def is_missing(x):
    return x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip().lower() in {"", "nan", "none"}
def norm_text(s):
    if is_missing(s): return None
    s=re.sub(r"[^\w\s]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip() or None
def norm_total(s):
    if is_missing(s): return None
    nums=re.findall(r"\d[\d,.]*",str(s).replace(" ",""))
    if not nums: return None
    tok=nums[-1]
    if re.search(r",\d{2}$",tok): tok=tok[:-3]+"."+tok[-2:]
    tok=tok.replace(",","")
    try: return f"{float(tok):.2f}"
    except ValueError: return None
def norm_date(s):
    if is_missing(s): return None
    core=re.sub(r"\s*([/\-.])\s*",r"\1",str(s).strip())
    for cand in (core,core.replace(" ","")):
        for fmt in DATE_FORMATS:
            try: return datetime.strptime(cand,fmt).strftime("%Y-%m-%d")
            except ValueError: pass
    return norm_text(core)
NORM={"company":norm_text,"address":norm_text,"date":norm_date,"total":norm_total}
def exact(pred,gold,field):
    return int(not is_missing(pred) and not is_missing(gold) and NORM[field](pred)==NORM[field](gold))

## 4. Tái lập baseline trên cùng tập mẫu

In [4]:
baseline_rows=[]
for _, r in df.iterrows():
    p=extract_baseline(r.ocr_text)
    row={"key":r.key,"split":r.split}
    for f in FIELDS:
        row[f+"_pred"]=p.get(f); row[f+"_gold"]=r[f]; row[f+"_em"]=exact(p.get(f),r[f],f)
    baseline_rows.append(row)
baseline=pd.DataFrame(baseline_rows)
def metrics(frame, split):
    sub=frame[frame.split==split]
    return pd.Series({**{f:sub[f+"_em"].mean() for f in FIELDS}, "MACRO_AVERAGE":np.mean([sub[f+"_em"].mean() for f in FIELDS])})
display(pd.DataFrame({s:metrics(baseline,s) for s in ["train","validation"]}).round(4))

,train,validation
company,0.2555,0.248
date,0.6946,0.752
address,0.2535,0.304
total,0.5329,0.568
MACRO_AVERAGE,0.4341,0.468


## 5. Phân tích nguyên nhân và quyết định cải tiến

Baseline cho thấy company/address thường vướng chọn sai dòng; total có nhiều dòng giá/GST; date vốn có điểm tốt hơn. Vì vậy, không thay tất cả các trường cùng lúc. Mỗi trường chỉ nhận luật mới khi có lợi trên Train; Validation dùng để xác nhận cuối.

In [7]:
# ===== LUẬT CẢI TIẾN COMPANY: xếp hạng tối đa 8 dòng đầu hóa đơn =====
COMPANY_HINTS = (
    "sdn", "bhd", "enterprise", "trading", "mart", "store", "restaurant",
    "cafe", "machinery", "gift", "home", "market", "supermarket", "shop",
)
ADDRESS_HINTS = (
    "jalan", "jln", "street", "road", "lorong", "taman", "lot",
    "selangor", "johor", "klang", "kuala lumpur", "penang", "perak", "melaka",
)
NON_COMPANY = re.compile(r"\b(invoice|receipt|cashier|tel|telephone|phone|fax|date|time|member|cash bill|tax invoice)\b", re.I)
DATE_LIKE = re.compile(r"\b\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}\b")
COMPANY_POSTCODE = re.compile(r"\b\d{5}\b")

def clean_company_line(value):
    value = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", value).strip(" \t|:;,-")

def company_candidate(lines):
    candidates = []
    for i, original in enumerate(lines[:8]):
        line = clean_company_line(original)
        if not (3 <= len(line) <= 90):
            continue
        if NON_COMPANY.search(line) or DATE_LIKE.search(line) or COMPANY_POSTCODE.search(line):
            continue
        if sum(char.isalpha() for char in line) < 3:
            continue
        low = line.lower()
        score = 0.0
        if any(hint in low for hint in COMPANY_HINTS):
            score += 2.5
        tokens = re.findall(r"[A-Za-z]{2,}", line)
        if len(tokens) >= 2:
            score += 0.8
        score += max(0.0, 0.7 - 0.12 * i)
        if any(hint in low for hint in ADDRESS_HINTS):
            score -= 2.0
        if re.search(r"\b(rm|qty|amount|price|total)\b", low):
            score -= 1.5
        candidates.append((score, -i, line))
    if not candidates:
        return None
    candidates.sort(reverse=True)
    return candidates[0][2]

def extract_improved(ocr_text):
    # Giữ nguyên 3 trường date/address/total từ Baseline.
    result = extract_baseline(ocr_text)
    lines = [clean_company_line(line) for line in str(ocr_text or "").replace("\x0c", "\n").splitlines() if clean_company_line(line)]
    company = company_candidate(lines)
    if company is not None:
        result["company"] = company
    return result

# Đánh giá riêng trên Train trước khi chốt luật company.
company_train = []
for _, row in df[df.split == "train"].iterrows():
    baseline_company = extract_baseline(row.ocr_text).get("company")
    improved_company = company_candidate([line.strip() for line in str(row.ocr_text).splitlines() if line.strip()]) or baseline_company
    company_train.append((exact(baseline_company, row.company, "company"), exact(improved_company, row.company, "company")))
company_train = pd.DataFrame(company_train, columns=["baseline_em", "candidate_em"])
print("Company Train Baseline:", round(company_train.baseline_em.mean(), 4))
print("Company Train Improved:", round(company_train.candidate_em.mean(), 4))
assert company_train.candidate_em.mean() > company_train.baseline_em.mean(), "Luật company chưa tốt hơn Baseline trên Train; cần xem xét lại trước khi chốt."

Company Train Baseline: 0.2555
Company Train Improved: 0.3313


## 6. Chạy phương pháp cải tiến đã chọn

Cấu hình đã chọn: chỉ thay company bằng xếp hạng các dòng đầu hóa đơn; date/address/total gọi nguyên luật baseline. Không dùng nhãn để dự đoán.

In [8]:
improved_rows=[]
for _,r in df.iterrows():
    p=extract_improved(r.ocr_text)
    row={"key":r.key,"split":r.split,"ocr_text_raw":r.ocr_text}
    for f in FIELDS:
        row[f+"_pred_raw"]=p.get(f)
        row[f+"_pred_norm"]=NORM[f](p.get(f))
        row[f+"_gold"]=r[f]
        row[f+"_gold_norm"]=NORM[f](r[f])
        row[f+"_em"]=exact(p.get(f),r[f],f)
    improved_rows.append(row)
improved=pd.DataFrame(improved_rows)
display(improved.head(3))

,key,split,ocr_text_raw,company_pred_raw,company_pred_norm,company_gold,company_gold_norm,company_em,date_pred_raw,date_pred_norm,...,address_pred_raw,address_pred_norm,address_gold,address_gold_norm,address_em,total_pred_raw,total_pred_norm,total_gold,total_gold_norm,total_em
0,X00016469612,train,tan woon yann\r\nBOOK TA -K (TAMAN DAYA) SDN B...,BOOK TA -K (TAMAN DAYA) SDN BHD,book ta k taman daya sdn bhd,BOOK TA .K (TAMAN DAYA) SDN BHD,book ta k taman daya sdn bhd,1,25/12/2018,2018-12-25,...,BOOK TA -K (TAMAN DAYA) SDN BHD NO.S: $557 & $...,book ta k taman daya sdn bhd no s 557 8 jalan ...,"NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 8...",no 53 55 57 59 jalan sagu 18 taman daya 81100 ...,0,91.00,91.00,9.00,9.00,0
1,X00016469619,train,"tan woon yann\r\nINDAH GIFT & HOME BECO\r\n27,...",INDAH GIFT & HOME BECO,indah gift home beco,INDAH GIFT & HOME DECO,indah gift home deco,0,19/10/2018,2018-10-19,...,"27, JALAN DEDAF 13, TAMAN JOHOR JAYA, 81100 JO...",27 jalan dedaf 13 taman johor jaya 81100 johor...,"27, JALAN DEDAP 13, TAMAN JOHOR JAYA, 81100 JO...",27 jalan dedap 13 taman johor jaya 81100 johor...,0,"60,34",60.34,60.30,60.30,0
2,X00016469620,train,tan woon yann\r\nMR O.1.Y. (JOHOR) SDN BHD\r\n...,MR O.1.Y. (JOHOR) SDN BHD,mr o 1 y johor sdn bhd,MR D.I.Y. (JOHOR) SDN BHD,mr d i y johor sdn bhd,0,12-01-19,2019-01-12,...,"MR O.1.Y. (JOHOR) SDN BHD LOT 1851-A & 1851-B,...",mr o 1 y johor sdn bhd lot 1851 a 1851 b jalan...,"LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERI...",lot 1851 a 1851 b jalan kpb 6 kawasan perindus...,0,"33,90",33.90,33.90,33.90,1


## 7. So sánh baseline và cải tiến trên cùng Validation

In [9]:
comparison=[]
for split in ["train","validation"]:
    b=baseline[baseline.split==split].set_index("key")
    i=improved[improved.split==split].set_index("key")
    for f in FIELDS:
        bm=b[f+"_em"].mean(); im=i[f+"_em"].mean()
        comparison.append({"split":split,"field":f,"baseline_em":bm,"improved_em":im,"delta":im-bm})
    comparison.append({"split":split,"field":"MACRO_AVERAGE","baseline_em":np.mean([b[f+"_em"].mean() for f in FIELDS]),"improved_em":np.mean([i[f+"_em"].mean() for f in FIELDS]),"delta":np.mean([i[f+"_em"].mean()-b[f+"_em"].mean() for f in FIELDS])})
comparison=pd.DataFrame(comparison)
display(comparison[comparison.split=="validation"].round(4))

,split,field,baseline_em,improved_em,delta
5,validation,company,0.248,0.320,0.072
6,validation,date,0.752,0.752,0.000
7,validation,address,0.304,0.304,0.000
8,validation,total,0.568,0.568,0.000
9,validation,MACRO_AVERAGE,0.468,0.486,0.018


## 8. Phân tích lỗi OCR và lỗi luật trích xuất

In [10]:
def gold_in_ocr(field,gold,text):
    if is_missing(gold): return False
    if field in ("company","address"):
        g=(norm_text(gold) or "").split(); t=(norm_text(text) or "").split()
        if not g: return False
        from collections import Counter
        return sum((Counter(g)&Counter(t)).values())/len(g)>=0.8
    if field=="date": return NORM[field](gold) in {NORM[field](m.group(0)) for m in DATE_RE.finditer(text)}
    return NORM[field](gold) in {NORM[field](x) for x in AMOUNT_RE.findall(text)}
for f in FIELDS:
    sub=improved[improved.split=="validation"]
    labels=[]
    for _,r in sub.iterrows():
        if r[f+"_em"]: labels.append("correct")
        elif gold_in_ocr(f,r[f+"_gold"],r.ocr_text_raw): labels.append("field_rule_error")
        else: labels.append("ocr_reading_error")
    print(f, pd.Series(labels).value_counts().to_dict())

company {'field_rule_error': 57, 'correct': 40, 'ocr_reading_error': 28}
date {'correct': 94, 'ocr_reading_error': 22, 'field_rule_error': 9}
address {'field_rule_error': 68, 'correct': 38, 'ocr_reading_error': 19}
total {'correct': 71, 'field_rule_error': 40, 'ocr_reading_error': 14}


## 9. Thử nghiệm tiền xử lý ảnh – giới hạn dữ liệu hiện có

Các file hiện được cung cấp ở vòng này chỉ gồm văn bản OCR CSV, không có ảnh gốc. Vì vậy không thể thực nghiệm và kết luận grayscale/resize/threshold/deskew. Muốn đo các bước này phải chạy lại OCR từ ảnh gốc trên Train/Validation, cùng engine và cấu hình OCR; không được suy diễn kết quả từ CSV văn bản.

## 10. Bảng giải thích cải tiến và kết luận

In [11]:
explanations=pd.DataFrame([
["company","Xếp hạng tối đa 8 dòng đầu; ưu tiên tín hiệu tên doanh nghiệp; loại dòng ngày/điện thoại/postcode/địa chỉ.","Giảm chọn nhầm tên người hoặc dòng đầu không phải cửa hàng.","Chấp nhận nếu Train và Validation cùng tăng."],
["date","Giữ nguyên regex và luật baseline.","Biến thể xếp hạng ngày chưa chứng minh lợi ích trên Train.","Không thay đổi."],
["address","Giữ nguyên luật baseline; không tự ghép dòng thiếu căn cứ.","Thử nghiệm ghép dòng làm giảm EM trên Train.","Không thay đổi."],
["total","Giữ nguyên luật baseline.","Chấm điểm nhiều ứng viên tiền làm tăng nguy cơ chọn nhầm giá/GST.","Không thay đổi."],
["image preprocessing","Chưa thực hiện do thiếu ảnh gốc trong đầu vào hiện tại.","Cần ảnh gốc và chạy lại OCR có kiểm soát.","Chưa có kết quả."],
],columns=["cải_tiến","cách_làm","lý_do","quyết_định"])
display(explanations)
# Xuất CSV theo split, không ghi đè baseline/GT.
for split,name in [("train","improved_train.csv"),("validation","improved_validation.csv")]:
    improved[improved.split==split].to_csv(OUT_DIR/name,index=False,encoding="utf-8-sig")
comparison.to_csv(OUT_DIR/"validation_and_train_comparison.csv",index=False,encoding="utf-8-sig")
explanations.to_csv(OUT_DIR/"improvement_explanations.csv",index=False,encoding="utf-8-sig")
print("Đã lưu kết quả tại:",OUT_DIR)

,cải_tiến,cách_làm,lý_do,quyết_định
0,company,Xếp hạng tối đa 8 dòng đầu; ưu tiên tín hiệu t...,Giảm chọn nhầm tên người hoặc dòng đầu không p...,Chấp nhận nếu Train và Validation cùng tăng.
1,date,Giữ nguyên regex và luật baseline.,Biến thể xếp hạng ngày chưa chứng minh lợi ích...,Không thay đổi.
2,address,Giữ nguyên luật baseline; không tự ghép dòng t...,Thử nghiệm ghép dòng làm giảm EM trên Train.,Không thay đổi.
3,total,Giữ nguyên luật baseline.,Chấm điểm nhiều ứng viên tiền làm tăng nguy cơ...,Không thay đổi.
4,image preprocessing,Chưa thực hiện do thiếu ảnh gốc trong đầu vào ...,Cần ảnh gốc và chạy lại OCR có kiểm soát.,Chưa có kết quả.


Đã lưu kết quả tại: c:\Users\Admin\HOA_DON_AI\outputs\improved
